# Validasi 32 lagu: graph GLU repair yang telah melewati batch iPhone

Batch iPhone yang dikirim menyelesaikan dua lagu x tiga pass dalam satu proses, tanpa crash yang tercatat, dengan label identik terhadap Mac dan seluruh prediksi finite. Ini bukti untuk enam run tersebut; belum bukti untuk seluruh lagu atau semua kondisi perangkat.

Notebook ini menjalankan **graph split-before-conv-v2** pada seluruh validation manual yang sama. Bobot palette6, head, konteks epoch 5, boundary, label dan pembagian dataset tetap. Tidak menjalankan training atau kuantisasi ulang. Report graph induk tetap digunakan sebagai pemeriksaan sumber; kandidat perbaikan dicatat terpisah dalam identitas cache sehingga hasil v1 tidak dapat dianggap hasil v2.

Full window 30/120 detik dijalankan ulang pada CPU Mac. Partial window menggunakan embedding CPU palette6 lama dari cache secara bitwise, seperti validasi induk. Dua lagu yang diuji di iPhone sudah memakai inferensi partial secara nyata; validasi Mac ini menilai kualitas seluruh 32 lagu secara terkontrol, bukan benchmark kecepatan iPhone. Test split tidak dipakai.

Jalankan cell berurutan. Tahap satu lagu memeriksa pipeline; tahap full memakai ulang hasil track yang sudah selesai. Tidak membuat salinan cache frame seluruh dataset. Tidak memasang model ke SongMap.


In [ ]:
from pathlib import Path
from datetime import datetime
import hashlib, json, subprocess

ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/validate_encoder_pipeline.py').is_file(), 'Buka notebook dari root SongMap'
SONGSTRUCT_ROOT=ROOT.parent/'Section-Annotations'
RUN_PYTHON=SONGSTRUCT_ROOT/'.venv-songformer311/bin/python'  # Jangan resolve symlink executable venv.
assert RUN_PYTHON.is_file(), f'Python tidak ditemukan: {RUN_PYTHON}'

# Report induk yang sama dengan batch iPhone yang telah ditinjau.
EXPECTED_PARENT_QUALITY='54b76055958a6b6f03702fc79664da65c7be4c4958be2e352964f58f374f3295'
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
PARENT=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-pipeline-quality-*/validation-all.json')
             if sha(p)==EXPECTED_PARENT_QUALITY),None)
assert PARENT, 'Report induk yang sesuai dengan batch iPhone belum ditemukan'
parent=json.loads(PARENT.read_text());parent_selection=json.loads((PARENT.parent/'selection.json').read_text())
assert parent['full_validation'] and parent['section_agreement']==1
EXPECTED_EXPORT_SHA=parent_selection['encoder_exports'];EXPORTS={}
for role,expected in EXPECTED_EXPORT_SHA.items():
    EXPORTS[role]=next((p.resolve() for p in (ROOT/'outputs.local').glob('encoder-acceleration-*/'+role+'/export-report.json') if sha(p)==expected),None)
    assert EXPORTS[role], f'Export induk {role} tidak ditemukan'

# Jika perlu, isi path repair-report.json yang dipakai pada aplikasi GLU Repair Anda.
REPAIR_REPORT=None
if REPAIR_REPORT:
    REPAIR=Path(REPAIR_REPORT).expanduser().resolve()
else:
    completed=[]
    for p in (ROOT/'outputs.local').glob('encoder-glu-repair-*/repair-report.json'):
        r=json.loads(p.read_text())
        if (r.get('protocol')=='encoder-glu-repair-v2' and r.get('policy')=='split-before-conv-v2'
                and r.get('parent_exports')==EXPECTED_EXPORT_SHA and set(r.get('roles',{}))=={'musicfm','muq'}):completed.append(p)
    assert completed, 'Repair selesai dengan parent yang cocok belum ditemukan'
    REPAIR=max(completed,key=lambda p:p.stat().st_mtime).resolve()
# Script akan memeriksa hash compiled assets, CPU parity dan payload pada kedua role/profile lagi.
expected_palette=parent_selection['context_identity']['palette_validation_sha256']
PALETTE_VALIDATION=next((p.resolve() for p in (ROOT/'outputs.local').glob('context-palette6-validation-*/validation-all.json') if sha(p)==expected_palette),None)
assert PALETTE_VALIDATION, 'Report palette6 konteks yang sesuai belum ditemukan'
selected=json.loads((PALETTE_VALIDATION.parent/'selection.json').read_text())
FROZEN=next((p.parent.resolve() for p in (ROOT/'outputs.local').glob('frozen-context-test-*/selection.json')
    if json.loads(p.read_text()).get('identity')==selected['frozen_selection_identity']),None)
assert FROZEN is not None, 'Snapshot konteks asli belum ditemukan'
BASELINE_POOL=ROOT/'outputs.local/section-context-palette6-cache-v1'
RESUME_OUTPUT=None  # Isi path encoder-glu-quality-* ini setelah restart kernel untuk melanjutkan.
OUTPUT=(Path(RESUME_OUTPUT).expanduser().resolve() if RESUME_OUTPUT else
    ROOT/'outputs.local'/('encoder-glu-quality-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f')))
subprocess.run([str(RUN_PYTHON),'-c','import sys,numpy,torch,coremltools,librosa; print("Python:",sys.executable); print("Torch:",torch.__version__); print("CoreMLTools:",coremltools.__version__)'],check=True)
print('Graph induk:',EXPORTS)
print('Repair kandidat:',REPAIR)
print('Pembanding:',PALETTE_VALIDATION)
print('Output:',OUTPUT)

def validate_quality(limit):
    command=[str(RUN_PYTHON),'-u',str(ROOT/'scripts/validate_encoder_pipeline.py'),
        '--songstruct-root',str(SONGSTRUCT_ROOT),'--songmap-root',str(ROOT),
        '--palette-validation',str(PALETTE_VALIDATION),'--frozen-evaluation',str(FROZEN),
        '--baseline-pool-cache',str(BASELINE_POOL),'--musicfm-export',str(EXPORTS['musicfm']),
        '--muq-export',str(EXPORTS['muq']),'--graph-repair',str(REPAIR),'--output-dir',str(OUTPUT),'--limit',str(limit)]
    log=Path(str(OUTPUT)+('.one.log' if limit else '.all.log'));log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8') as handle:
        process=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Validation gagal ({status}); lihat {log}. Hasil track yang sudah lengkap dapat dipakai ulang.')


In [ ]:
# Satu lagu dahulu. Ini smoke check, belum ukuran kualitas seluruh validation.
validate_quality(1)


In [ ]:
# Seluruh validation, termasuk semua target manual layak pada tiap lagu.
# Hasil track pertama dipakai ulang jika identitasnya cocok; tidak menjalankan training.
validate_quality(0)


In [ ]:
from IPython.display import display, Markdown
report=json.loads((OUTPUT/'validation-all.json').read_text())
assert report['full_validation'] and not report['test_used']
assert report['static_graph_policy']=='split-before-conv-v2'
assert report['graph_repair_sha256']==sha(REPAIR)
assert report['plan_id']==parent['plan_id'] and report['provenance_id']==parent['provenance_id']
lines=['| Pembanding/kandidat | Akurasi | Macro F1 |','|---|---|---|']
for name,key in [('Konteks pada fitur training asli','original_training_features_context'),
                 ('Pipeline palette6 saat ini','baseline_palette6_context'),('Encoder GLU repair + partial lama','candidate')]:
    r=report[key]
    lines.append(f"| {name} | {r['section_accuracy']:.2%} | {r['macro_f1_present']:.6f} |")
display(Markdown('\n'.join(lines)))
for key in ['comparison_to_current_pipeline','comparison_to_original_context']:
    print('\n',key,':',json.dumps(report[key],indent=2))
print('\nKesamaan label dengan pipeline sekarang:',f"{report['section_agreement']:.2%}")
print('Jumlah section yang berubah:',len(report['changed_sections']))
print('Jalur window:',report['routes'])
print('Per-class F1 kandidat:',{k:round(v['f1'],4) for k,v in report['candidate']['per_class'].items()})
print('\nKirim file:',OUTPUT/'validation-all.json')

print('Graph induk F1:',parent['candidate']['macro_f1_present'])
print('Delta F1 repair vs graph induk:',report['candidate']['macro_f1_present']-parent['candidate']['macro_f1_present'])
print('Report kualitas dan cache ini tidak otomatis mengaktifkan routing aplikasi.')


Hasil yang perlu dikirim: `validation-all.json` dari folder `encoder-glu-quality-*`. Periksa per-class F1 dan section yang berubah, termasuk kelas minoritas; label agreement terhadap Mac dalam batch iPhone bukan akurasi terhadap anotasi manual.

Setiap report/cache terikat hash parent, repair, compiled model dan script. Jika interupsi, jalankan kembali cell yang sama. Setelah restart kernel, isi `RESUME_OUTPUT` dengan folder hasil validasi ini. Source yang berubah akan ditolak; jangan gunakan hasil v1 sebagai cache v2.

Setelah kualitas graph perbaikan diperiksa, target optimasi kecepatan berikutnya adalah convolution frontend dan biaya partial CPU. Pada batch yang dikirim, dua convolution frontend masih menggunakan sekitar 40–44% total waktu; head hanya sekitar 0,6–2 detik dan konteks beberapa milidetik. Pembandingan GPU/CPU yang seimbang serta pengemasan tanpa duplikasi bobot tetap diperlukan sebelum produksi. Notebook ini tidak mengubah frontend, routing produksi, ukuran input, boundary atau checkpoint.

Jika report repair lolos dan hasil batch iPhone tetap sesuai, lanjutkan [frontend_routes_iphone.ipynb](frontend_routes_iphone.ipynb) untuk pembandingan compute route convolution frontend. Notebook tersebut mengunci hasil kualitas yang telah ditinjau dan tidak melakukan training atau perubahan bobot.
